In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 9 — Ejercicio 1
# ---------------------------------------------------------------

from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

california = fetch_california_housing()
X, y = california.data, california.target

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

for cv in [3, 5, 10]:
    scores = cross_val_score(rf, X, y, cv=cv, scoring='r2', n_jobs=-1)
    print(f"cv={cv:2d} | R² = {scores.mean():.4f} ± {scores.std():.4f}")


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 9 — Ejercicio 2
# ---------------------------------------------------------------

from sklearn.datasets import load_digits
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

X_dig, y_dig = load_digits(return_X_y=True)
X_dig = X_dig / 16.0   # normalizar a [0, 1]

param_dist = {
    "C":     loguniform(0.1, 100),
    "gamma": loguniform(0.001, 0.1),
}

semilla = 42
rs = RandomizedSearchCV(
    SVC(kernel="rbf"),
    param_dist,
    n_iter=20,
    cv=3,
    scoring='accuracy',
    random_state=semilla,
    n_jobs=-1,
)
rs.fit(X_dig, y_dig)
print(f"Mejores params: {rs.best_params_}")
print(f"Mejor CV score: {rs.best_score_:.4f}")


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 9 — Ejercicio 3
# ---------------------------------------------------------------

import pandas as pd

# pipe y X_train/y_train provienen del Cap. 6 (pipeline Titanic)
param_grid_pipe = {
    "clf__n_estimators":    [100, 200, 300],
    "clf__max_depth":       [None, 5, 10],
    "clf__min_samples_leaf": [1, 3],
    "prep__num__strategy":  ["mean", "median"],
}

from sklearn.model_selection import GridSearchCV
gs = GridSearchCV(
    pipeline_tit, param_grid_pipe,
    cv=5, scoring="accuracy", n_jobs=-1
)
gs.fit(X_tit, y_tit)

resultados = pd.DataFrame(gs.cv_results_)

# 1. Top 5 combinaciones
top5 = (resultados
        .sort_values("mean_test_score", ascending=False)
        [["param_clf__n_estimators", "param_clf__max_depth", "mean_test_score"]]
        .head(5))
print("Top 5 combinaciones:")
print(top5.to_string(index=False))

# 2. Media por max_depth
print("\nScore medio por max_depth:")
print(resultados.groupby(
    "param_clf__max_depth")["mean_test_score"].mean().round(4))

# 3. Impacto de estrategia de imputación
print("\nScore medio por estrategia de imputación:")
print(resultados.groupby(
    "param_prep__num__strategy",
)["mean_test_score"].mean().round(4))